# Analise de Dados - Camada Gold
Este notebook responde as perguntas de negocio e gera a camada Gold agregada (VIEW e TABELA), com JOIN + GROUP BY.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import banco
import warnings
from IPython.display import Markdown
warnings.filterwarnings('ignore')

conexao = banco.conectar()
sns.set_theme(style='whitegrid')


### Parte 1: Camada Silver - Analises Diretas
As 3 primeiras perguntas usam apenas `silver_viagem` (uma unica tabela, sem risco de fan-out ao agregar por `nome_orgao_superior`, `destinos` ou `duracao_dias`).

In [ ]:
# 1. Os 5 orgaos com maior custo total?
q1 = "SELECT nome_orgao_superior, SUM(valor_total) as custo FROM silver_viagem GROUP BY nome_orgao_superior ORDER BY custo DESC LIMIT 5"
df1 = pd.read_sql(q1, conexao)
display(df1)
plt.figure(figsize=(10,4))
sns.barplot(data=df1, y='nome_orgao_superior', x='custo', palette='viridis')
plt.title('Top 5 Orgaos com Maior Custo Total')
plt.xlabel('Custo Total (R$)')
plt.ylabel('Orgao Superior')
plt.show()
display(Markdown(
    f"**Interpretacao:** o orgao **{df1.iloc[0]['nome_orgao_superior']}** lidera o gasto total com viagens "
    f"(R$ {df1.iloc[0]['custo']:,.2f}), concentrando a maior fatia entre os 5 principais orgaos. "
    f"Isso indica onde a fiscalizacao e o controle orcamentario de viagens deveriam ser priorizados."
))

# 2. Os 3 destinos com maior custo medio por viagem?
q2 = "SELECT destinos, AVG(valor_total) as custo_medio FROM silver_viagem GROUP BY destinos ORDER BY custo_medio DESC LIMIT 3"
df2 = pd.read_sql(q2, conexao)
display(df2)
plt.figure(figsize=(10,4))
sns.barplot(data=df2, y='destinos', x='custo_medio', palette='magma')
plt.title('Top 3 Destinos com Maior Custo Medio')
plt.xlabel('Custo Medio (R$)')
plt.ylabel('Destinos')
plt.show()
display(Markdown(
    f"**Interpretacao:** viagens para **{df2.iloc[0]['destinos']}** tem o maior custo medio "
    f"(R$ {df2.iloc[0]['custo_medio']:,.2f} por viagem). Esse destino merece atencao especial no planejamento "
    f"orcamentario, seja por distancia, logistica ou custo de vida local."
))

# 3. A viagem de maior duracao e seu custo total?
q3 = "SELECT id_viagem, duracao_dias, valor_total FROM silver_viagem ORDER BY duracao_dias DESC LIMIT 1"
df3 = pd.read_sql(q3, conexao)
display(df3)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(['Duracao'], df3['duracao_dias'], color='steelblue')
axes[0].set_title('Duracao (dias)')
axes[0].set_xlabel('Viagem ' + str(df3['id_viagem'].iloc[0]))
axes[0].set_ylabel('Dias')
axes[1].bar(['Custo Total'], df3['valor_total'], color='darkorange')
axes[1].set_title('Custo Total (R$)')
axes[1].set_xlabel('Viagem ' + str(df3['id_viagem'].iloc[0]))
axes[1].set_ylabel('Valor (R$)')
plt.suptitle('Viagem de Maior Duracao')
plt.tight_layout()
plt.show()
display(Markdown(
    f"**Interpretacao:** a viagem **{df3.iloc[0]['id_viagem']}** foi a mais longa do periodo, com "
    f"**{df3.iloc[0]['duracao_dias']} dias** de duracao e custo total de **R$ {df3.iloc[0]['valor_total']:,.2f}**. "
    f"Viagens muito longas merecem revisao de justificativa e comparacao do custo por dia com a media geral."
))


### Parte 2: Construcao da Camada Gold Agregada

**Correcao de design:** a primeira versao deste notebook resolvia o fan-out (produto cartesiano) evitando a camada Gold nas perguntas 4 a 7 e consultando `silver_pagamento`/`silver_trecho` diretamente. Isso contraria o requisito do projeto, que exige que as perguntas de negocio sejam respondidas **a partir da camada Gold**.

A forma correta de evitar o fan-out **sem abandonar a Gold** e nao construir uma unica tabela achatada por `id_viagem` (que forcaria o JOIN viagem x pagamento x trecho a duplicar linhas). Em vez disso, a Gold e composta por **agregados na granularidade de cada pergunta de negocio** 鈥� pratica padrao em Arquitetura Medallion, onde a camada Gold e um conjunto de marts prontos para consumo, e nao uma unica tabela universal.

Sao criadas 2 VIEWs + 2 TABELAS fisicas Gold, cada uma com JOIN + GROUP BY:
- `gold_resumo_viagens`: 1 linha por viagem (usada nas perguntas 1 a 3 e como visao geral).
- `gold_pagamentos_por_tipo`: agregado por `tipo_pagamento` (JOIN viagem+pagamento, GROUP BY tipo) 鈥� usada na pergunta 4 e 7.
- `gold_trechos_por_transporte_uf`: agregado por `meio_transporte` e `destino_uf` (JOIN viagem+trecho, GROUP BY) 鈥� usada nas perguntas 5 e 6.

In [ ]:
# --- Gold 1: resumo por viagem (pagamento e trecho pre-agregados por id_viagem
# ANTES do JOIN com viagem, para nao duplicar linhas quando uma viagem tem mais
# de um pagamento e mais de um trecho ao mesmo tempo) ---
sql_gold_view_1 = """
CREATE OR REPLACE VIEW gold_resumo_viagens AS
SELECT
    v.id_viagem,
    v.nome_orgao_superior,
    v.valor_total                   AS custo_viagem,
    v.duracao_dias,
    COALESCE(pag.total_pago, 0)     AS total_pago,
    COALESCE(pag.qtd_pagamentos, 0) AS qtd_pagamentos,
    COALESCE(tre.qtd_trechos, 0)    AS qtd_trechos
FROM silver_viagem v
LEFT JOIN (
    SELECT id_viagem, SUM(valor) AS total_pago, COUNT(*) AS qtd_pagamentos
    FROM silver_pagamento
    GROUP BY id_viagem
) pag ON pag.id_viagem = v.id_viagem
LEFT JOIN (
    SELECT id_viagem, COUNT(*) AS qtd_trechos
    FROM silver_trecho
    GROUP BY id_viagem
) tre ON tre.id_viagem = v.id_viagem;
"""
banco.executar(conexao, sql_gold_view_1)
banco.executar(conexao, "DROP TABLE IF EXISTS gold_resumo_viagens_tab;")
banco.executar(conexao, "CREATE TABLE gold_resumo_viagens_tab AS SELECT * FROM gold_resumo_viagens;")
print("gold_resumo_viagens (VIEW + TABELA) criada com sucesso!")

# --- Gold 2: pagamentos agregados por tipo_pagamento e por orgao pagador ---
# JOIN com silver_viagem para trazer contexto de orgao/viagem, GROUP BY no
# nivel de pagamento (nao ha trecho envolvido aqui, entao nao ha fan-out).
sql_gold_view_2 = """
CREATE OR REPLACE VIEW gold_pagamentos_por_tipo AS
SELECT
    p.tipo_pagamento,
    p.nome_orgao_pagador,
    v.nome_orgao_superior,
    COUNT(*)          AS qtd_pagamentos,
    SUM(p.valor)       AS valor_total,
    AVG(p.valor)        AS valor_medio
FROM silver_pagamento p
JOIN silver_viagem v ON v.id_viagem = p.id_viagem
GROUP BY p.tipo_pagamento, p.nome_orgao_pagador, v.nome_orgao_superior;
"""
banco.executar(conexao, sql_gold_view_2)
banco.executar(conexao, "DROP TABLE IF EXISTS gold_pagamentos_por_tipo_tab;")
banco.executar(conexao, "CREATE TABLE gold_pagamentos_por_tipo_tab AS SELECT * FROM gold_pagamentos_por_tipo;")
print("gold_pagamentos_por_tipo (VIEW + TABELA) criada com sucesso!")

# --- Gold 3: trechos agregados por meio_transporte e destino_uf ---
# JOIN com silver_viagem para contexto, GROUP BY no nivel de trecho (nao ha
# pagamento envolvido aqui, entao tambem nao ha fan-out).
sql_gold_view_3 = """
CREATE OR REPLACE VIEW gold_trechos_por_transporte_uf AS
SELECT
    t.meio_transporte,
    t.destino_uf,
    v.nome_orgao_superior,
    COUNT(*) AS qtd_trechos
FROM silver_trecho t
JOIN silver_viagem v ON v.id_viagem = t.id_viagem
GROUP BY t.meio_transporte, t.destino_uf, v.nome_orgao_superior;
"""
banco.executar(conexao, sql_gold_view_3)
banco.executar(conexao, "DROP TABLE IF EXISTS gold_trechos_por_transporte_uf_tab;")
banco.executar(conexao, "CREATE TABLE gold_trechos_por_transporte_uf_tab AS SELECT * FROM gold_trechos_por_transporte_uf;")
print("gold_trechos_por_transporte_uf (VIEW + TABELA) criada com sucesso!")

display(pd.read_sql("SELECT * FROM gold_resumo_viagens_tab LIMIT 5", conexao))


### Parte 3: Camada Gold - Analises Agregadas
Todas as perguntas a seguir consultam exclusivamente as tabelas/VIEWs **Gold** criadas na Parte 2 鈥� nenhuma consulta direta em `silver_pagamento` ou `silver_trecho`.

In [ ]:
# 4. Qual o tipo de pagamento com maior valor medio?
# Consulta a partir da Gold (gold_pagamentos_por_tipo_tab), reagregando por tipo_pagamento
q4 = """
SELECT tipo_pagamento, SUM(valor_total) AS valor_total, SUM(qtd_pagamentos) AS qtd_pagamentos,
       SUM(valor_total) / SUM(qtd_pagamentos) AS valor_medio
FROM gold_pagamentos_por_tipo_tab
GROUP BY tipo_pagamento
ORDER BY valor_medio DESC
LIMIT 5
"""
df4 = pd.read_sql(q4, conexao)
display(df4)
display(Markdown(
    f"**Interpretacao:** o tipo de pagamento **{df4.iloc[0]['tipo_pagamento']}** tem o maior valor medio por "
    f"lancamento (R$ {df4.iloc[0]['valor_medio']:,.2f}), sendo a categoria de despesa que mais pesa individualmente "
    f"em cada pagamento."
))

# 5. Qual o meio de transporte mais usado nos trechos?
# Consulta a partir da Gold (gold_trechos_por_transporte_uf_tab)
q5 = """
SELECT meio_transporte, SUM(qtd_trechos) AS qtd
FROM gold_trechos_por_transporte_uf_tab
GROUP BY meio_transporte
ORDER BY qtd DESC
"""
df5 = pd.read_sql(q5, conexao)
display(df5)
plt.figure(figsize=(10,4))
sns.barplot(data=df5, x='meio_transporte', y='qtd', palette='coolwarm')
plt.title('Meios de Transporte Mais Utilizados')
plt.xlabel('Meio de Transporte')
plt.ylabel('Quantidade de Trechos')
plt.show()
display(Markdown(
    f"**Interpretacao:** **{df5.iloc[0]['meio_transporte']}** e o meio de transporte predominante nos trechos "
    f"({df5.iloc[0]['qtd']} ocorrencias), o que pode orientar futuras negociacoes de contratos corporativos "
    f"com fornecedores desse setor."
))

# 6. Qual UF de destino aparece em mais trechos?
# Consulta a partir da Gold (gold_trechos_por_transporte_uf_tab)
q6 = """
SELECT destino_uf, SUM(qtd_trechos) AS qtd
FROM gold_trechos_por_transporte_uf_tab
GROUP BY destino_uf
ORDER BY qtd DESC
LIMIT 5
"""
df6 = pd.read_sql(q6, conexao)
display(df6)
plt.figure(figsize=(10,4))
sns.barplot(data=df6, x='destino_uf', y='qtd', palette='crest')
plt.title('Top 5 UFs de Destino')
plt.xlabel('UF (Unidade Federativa)')
plt.ylabel('Quantidade de Trechos')
plt.show()
display(Markdown(
    f"**Interpretacao:** **{df6.iloc[0]['destino_uf']}** e a UF de destino mais frequente nos trechos "
    f"({df6.iloc[0]['qtd']} ocorrencias), revelando a principal rota logistica das viagens a servico no periodo."
))

# 7. Qual orgao pagou mais no total?
# Consulta a partir da Gold (gold_pagamentos_por_tipo_tab)
q7 = """
SELECT nome_orgao_pagador, SUM(valor_total) AS total_pago
FROM gold_pagamentos_por_tipo_tab
GROUP BY nome_orgao_pagador
ORDER BY total_pago DESC
LIMIT 5
"""
df7 = pd.read_sql(q7, conexao)
display(df7)
plt.figure(figsize=(10,4))
sns.barplot(data=df7, y='nome_orgao_pagador', x='total_pago', palette='rocket')
plt.title('Top 5 Orgaos Pagadores pelo Total Pago')
plt.xlabel('Valor Total Pago (R$)')
plt.ylabel('Orgao Pagador')
plt.show()
display(Markdown(
    f"**Interpretacao:** **{df7.iloc[0]['nome_orgao_pagador']}** foi o orgao pagador com maior volume de "
    f"recursos desembolsados (R$ {df7.iloc[0]['total_pago']:,.2f}), sendo o principal responsavel financeiro "
    f"pelas viagens no periodo analisado."
))
